In [ ]:
import kaggle_benchmarks as kbench
import json
import re
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

CANONICAL_NUMERIC_ANSWER = 137
ABS_TOL = 1.0  # nearest integer target

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text:
        return None

    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start = text.find("{")
        end = text.rfind("}")
        if start == -1 or end == -1 or end <= start:
            return None
        blob = text[start:end + 1]

    try:
        return json.loads(blob)
    except Exception:
        return None

def safe_get_attr(obj, attr_name, default=None):
    try:
        return getattr(obj, attr_name, default)
    except Exception:
        return default

def build_trace(
    *,
    task_id,
    llm,
    prompt,
    response,
    parsed,
    final_answer,
    normalized_answer,
    passed,
    failure_mode,
):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "normalized_answer": normalized_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt,
        "tokens_input": safe_get_attr(llm, "last_input_tokens"),
        "tokens_output": safe_get_attr(llm, "last_output_tokens"),
        "cost": safe_get_attr(llm, "last_cost"),
        "latency_ms": safe_get_attr(llm, "last_latency_ms"),
    }

def normalize_numeric_answer(text):
    if text is None:
        return ""

    s = str(text).strip().lower()
    replacements = {
        "$": "",
        ",": " ",
        "dots per": "",
        "per": "/",
        "μ": "u",
        "µ": "u",
        "\\mu": "u",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)

    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_first_number(text):
    s = normalize_numeric_answer(text)
    if not s:
        return None, s

    m = re.search(r"[-+]?\d+(?:\.\d+)?", s)
    if not m:
        return None, s

    try:
        return float(m.group(0)), s
    except Exception:
        return None, s

def code_verifier(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return False, "hallucination", normalized

    if abs(value - CANONICAL_NUMERIC_ANSWER) <= ABS_TOL:
        return True, None, normalized

    return False, None, normalized

def classify_failure_fp_0005(answer_text):
    value, normalized = extract_first_number(answer_text)

    if value is None:
        return "hallucination"

    # Uploaded task-specific dominant failure
    if abs(value - 54) <= 1.0:
        return "misapplication_of_equation_or_model"

    # Near miss -> arithmetic / rounding
    if abs(value - CANONICAL_NUMERIC_ANSWER) <= 10:
        return "calculation_error"

    # Otherwise likely wrong physical normalization or model application
    return "misapplication_of_equation_or_model"

# ----------------------------
# Frontier Physics Task 005
# ----------------------------
@kbench.task(
    name="FP-0005 Nanodot LSPR Density for Absorption Target",
    description="Hard electromagnetism task on quasistatic plasmonic nanodot absorption and required number density in a dielectric host."
)
def fp_0005_nanodot_lspr_density(llm) -> tuple[int, int]:
    prompt = r"""You are solving a physics problem. Return valid JSON only — no prose outside the JSON.

An anti-counterfeit security thread is fabricated by embedding a dilute, random population of identical metallic nanodots into a lossless polymer film. The nanodots are spherical with radius $r=10\mathrm{nm}$ and are sufficiently dilute that inter-dot coupling is negligible; treat the thread’s total absorption cross section as the sum of the absorption cross sections of the individual dots.

At the design vacuum wavelength $\lambda_0=515\mathrm{nm}$, the metal’s complex relative permittivity is $\epsilon=-13.005+0.327i$ and its Drude plasma frequency is $f_p=2.18\times10^{15}\mathrm{Hz}$. The polymer is chosen so that the dots’ localized surface plasmon resonance occurs at $\lambda_0$ for a quasistatic sphere in a dielectric host. Assume the simplified Drude model and the dipole-limit absorption-efficiency expression; ignore scattering.

A $1\mu\mathrm{m}^2$ patch of the security thread has physical thickness $0.80\mu\mathrm{m}$, so the patch volume is $0.80\mu\mathrm{m}^3$. The design target is a maximum overall absorption cross section of $1\mu\mathrm{m}^2$ for that $1\mu\mathrm{m}^2$ illuminated patch.

Question: What nanodot number density, in dots per $\mu\mathrm{m}^3$, expressed as the nearest integer, is required to meet the target?

Return JSON only in the following format:
{
  "final_answer": "<nearest integer density>"
}"""

    response = llm.prompt(prompt)
    parsed = extract_json(response)

    total_checks = 1
    passed_checks = 0
    final_answer = ""
    normalized_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")
        code_result, code_failure, normalized_answer = code_verifier(final_answer)

        if code_result is True:
            passed_checks = 1
        else:
            failure_mode = code_failure or classify_failure_fp_0005(final_answer)

    trace = build_trace(
        task_id="fp_0005",
        llm=llm,
        prompt=prompt,
        response=response,
        parsed=parsed,
        final_answer=final_answer,
        normalized_answer=normalized_answer,
        passed=(passed_checks == 1),
        failure_mode=failure_mode,
    )
    TRACE_LOG.append(trace)

    return (passed_checks, total_checks)

In [ ]:
fp_0005_nanodot_lspr_density.run(kbench.llm)

In [ ]:
results = fp_0005_nanodot_lspr_density.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd

trace_df = pd.DataFrame(TRACE_LOG)
trace_df[trace_df["task_id"] == "fp_0005"]